# Pilot XAI-DSS trên IndPenSim

Notebook này chạy lại toàn bộ thí nghiệm ban đầu **trên máy chủ của Google**. Máy tính của bạn chỉ hiển thị trang web.

**Trước khi chạy:**
1. Upload `pilot_colab.zip` và notebook này vào cùng một thư mục trên Google Drive, ví dụ `MyDrive/De_Tai_Nong_San/`.
2. Chọn *Runtime → Change runtime type → CPU*. Không cần GPU.
3. Chạy lần lượt từng ô từ trên xuống.

**Nếu bị ngắt kết nối:** chạy lại các ô 1–3, rồi chạy lại ô đang dở. Những mẻ đã xong được lưu trên Drive và sẽ được bỏ qua.

## Ô 1. Kết nối Google Drive và giải nén code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
DRIVE_DIR = '/content/drive/MyDrive/De_Tai_Nong_San'   # sửa nếu bạn để file ở thư mục khác
ZIP_PATH = os.path.join(DRIVE_DIR, 'pilot_colab.zip')
WORK = os.path.join(DRIVE_DIR, 'pilot')

if not os.path.exists(os.path.join(WORK, 'sim.py')):
    zipfile.ZipFile(ZIP_PATH).extractall(DRIVE_DIR)
    print('Đã giải nén vào', WORK)
os.chdir(WORK)
print('Thư mục làm việc:', os.getcwd())
print(sorted(os.listdir('.')))

## Ô 2. Cài thư viện (thường Colab đã có sẵn)

In [ ]:
!pip install -q lightgbm shap
import numpy, scipy, pandas, sklearn, lightgbm, shap
print('numpy', numpy.__version__, '| scipy', scipy.__version__, '| lightgbm', lightgbm.__version__, '| shap', shap.__version__)

## Ô 3. Chọn quy mô chạy

| Chế độ | Mẻ huấn luyện | Mẻ kiểm thử | Thời gian (2 CPU) |
|---|---|---|---|
| `NHANH` | 50 | 10 | khoảng 45–75 phút |
| `DAY_DU` | 120 | 24 | khoảng 2–3 giờ |

Nên chạy `NHANH` trước. Nếu kết quả có triển vọng thì chạy tiếp `DAY_DU`: các mẻ đã xong sẽ được dùng lại, không chạy lại.

In [ ]:
CHE_DO = 'NHANH'          # 'NHANH' hoặc 'DAY_DU'
N_TRAIN, N_TEST = (50, 10) if CHE_DO == 'NHANH' else (120, 24)
print('Chế độ:', CHE_DO, '| mẻ huấn luyện:', N_TRAIN, '| mẻ kiểm thử:', N_TEST, '| số CPU:', os.cpu_count())

## Ô 4. Kiểm tra nhanh bộ mô phỏng (khoảng 1 phút)

Chạy 3 mẻ theo công thức. Kết quả trên máy cục bộ: seed 1 ≈ **3640 kg**, seed 2 ≈ **3258 kg**, seed 3 ≈ **2705 kg**. Thể tích bình phải luôn dưới 100.000 L. Sai khác nhỏ do phiên bản thư viện là bình thường.

In [ ]:
!python -W ignore sim.py

## Ô 5. Sinh dữ liệu huấn luyện (phần lâu nhất)

In [ ]:
!python -W ignore gen_data.py --seeds {N_TRAIN}

## Ô 6. Huấn luyện mô hình dự báo (vài giây)

Kiểm định chéo chia theo mẻ: không mẻ nào vừa nằm trong tập huấn luyện vừa nằm trong tập kiểm định.

In [ ]:
!python train_dss.py

## Ô 7. Chạy DSS vòng kín và kiểm chứng lời giải thích bằng mô phỏng

In [ ]:
!python -W ignore eval_dss.py --seeds {N_TEST}

## Ô 8. Tổng hợp số liệu

In [ ]:
!python analyze.py

## Ô 9. Thí nghiệm phụ trên dữ liệu thịt bò thật (vài phút)

In [ ]:
!python beef_pilot.py

## Ô 10. Gom kết quả thành `ket_qua_pilot.zip` trên Drive

Tải file này về máy và đặt vào `D:\De_Tai_Nong_San\`, rồi báo cho Claude để phân tích và viết báo cáo cho giáo sư.

In [ ]:
out = os.path.join(DRIVE_DIR, 'ket_qua_pilot.zip')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in ['results', 'models']:
        for root, _, files in os.walk(folder):
            for f in files:
                if '/parts' not in root:
                    z.write(os.path.join(root, f))
    if os.path.exists('data/train.csv'):
        z.write('data/train.csv')
print('Đã lưu', out, f'({os.path.getsize(out)/1e6:.1f} MB)')